## DSAN 6000 Homework 3B: Map-Reduce for Efficient Matrix Operations

## Overview

You made it to the first DSAN 6000 homework introducing a new coding concept! The goal of this part is for you to gain hands-on experience with how **Map-Reduce** allows paralellization of **non-embarrassingly-parallel** algorithms.

There is really only a single "task" here, split into two subparts: the task of comparing the performance of the **matrix-vector multiplication algorithm** you learned in school with the performance of a **map-reduced version** of the same algorithm. On your EC2 instance, you should be able to speed the algorithm up by 200% to 300%! And in fact, this speedup – the ratio of total time required to run serial and parallel versions – grows as the size of the matrix grows (since you're moving from $\overline{O}(n^3)$ to $\overline{O}(n^2)$)

Note that the sections labeled as **"Part"**, like **Part 2.2** below, are cases where you can just run the cell for free points! (Meaning, you do not need to write any new code – as long as the provided code in these cells runs successfully, you have completed that part) The portions requiring you to write new code are labeled **Question** rather than **Part**

## Part 1: "Classical" Matrix\-Vector Multiplication

The "by hand" matrix-vector multiplication algorithm you probably learned at some point in Algebra or Linear Algebra looks as follows:

<center>
<img src='https://raw.githubusercontent.com/jpowerj/dsan-content/refs/heads/main/2025-spr-dsan5500/hw5/matvec_3x3.png' width='75%' />
</center>

If it helps at all, Jeff's brain encodes multiplying a matrix $\mathbf{A}$ by a vector $\mathbf{b}$ as something like:

* Taking the inner products of each row vector in $\mathbf{A}$ with $\mathbf{b}$, where
* The **row index** of the inner product within the resulting vector will match the **row index** that we're currently looking at in $\mathbf{A}$

In other words, if I was computing the product pictured above on paper, I would start by drawing a big "box" (blank space for the result vector) divided into slots, like the following (where the numbers are just there to make it easier to reference the slots):

$$
\mathbf{A}\cdot \mathbf{b} = \left[
\begin{array}{c}
1 \\\hline
2 \\\hline
3
\end{array}
\right]
$$

and then I'd go slot-by-slot, saying e.g. when I came to **Slot 3** that:

*This slot is in the **third** row of the resulting vector, therefore it should contain the inner product of the **third row** of $\mathbf{A}$ and $\mathbf{b}$*


### Question 1\.1

So, to enable us to compare the efficiency of this "classical" algorithm with a map-reduced version, implement the function started for you, `mat_vec_product(A, b)`, in the following code cell. Note that:

* The "matrix" `A` that your function takes as its first argument is just a Python `list` where each element is itself a Python `list` of `float` values.
* The "vector" `b` that that your function takes as its second argument is just a Python `list` where each element is a `float` value.
* You **can** assume that `A` and `b` will be **conformable**:
  * `A` will be a square ($n \times n$) matrix
  * The number of elements in `b` will be the same $n$
  * `A` and `b` will have valid (integer) numeric entries ($A \in GL_n(\mathbb{R})$, $b \in \mathbb{R}^n$), and so on. More simply: our focus here is on the efficiency gains from parallelization, not type-checking, so you **don't** need to do any validation on the inputs.
* You should **not** use any functions from external libraries like **NumPy** (since these will call the [BLAS routines](https://www.netlib.org/blas/) on the EC2 instance's processor, which are hyper-efficiently parallelized), only the built-in Python operators like `*` and `+`

In [ ]:
#| label: Q1.1-response
def mat_vec_product(A: list[list[int]], b: list[int]) -> list[int]:
  n = len(A)
  # Construct a length-n vector of 0s, to be filled with the results
  c = [0 for _ in range(n)]
  # Your code here: Implement serial matrix-vector multiplication via a for loop
  # over the rows of the argument A

  return c


In [ ]:
#| label: Q1.1-public-test
# Test that your implementation worked

A_small = [[3,2,0],[0,4,1],[2,0,1]]
b_small = [4, 3, 1]

result = mat_vec_product(A_small, b_small)
print(f"Function call result: {result}")
prod_expected = [18, 13, 9]
print(f"Expected result: {prod_expected}")

Function call result: [0, 0, 0]
Expected result: [18, 13, 9]


### Part 1\.2: Measuring Runtime

Now let's use our function to multiply a large matrix and vector, and get the runtime as a baseline against which we can compare our Map-Reduced version.

Run the following code cell to generate a $2500 \times 2500$ matrix $\mathbf{A}$ and a length-2500 vector $\mathbf{b}$. Note that here we **do** use `numpy` finally, but only to randomly-generate the input matrix and vector. Once these are generated, they're converted into Python lists, so that we don't trigger the BLAS calls during the actual multiplication.

In [ ]:
#| label: Q1.2-gen-matrix
import os
from datetime import datetime, timedelta
import numpy as np
import joblib
rng = np.random.default_rng(seed=6000)
rand_n = 2500
A_large = rng.integers(low=-100, high=100, size=(rand_n,rand_n))
b_large = rng.integers(low=-100, high=100, size=rand_n)
# Ensure the data subdirectory exists
os.makedirs("data", exist_ok=True)
joblib.dump(A_large, "data/A_large.pkl")
joblib.dump(b_large, "data/b_large.pkl")
# We also save A_small and b_small, for use when we test the Map-Reduce implementation below
joblib.dump(A_small, "data/A_small.pkl")
joblib.dump(b_small, "data/b_small.pkl")

['b_small.pkl']

Once `A_large.pkl` and `b_large.pkl` have been generated and saved, we can test how long your serial implementation from above takes to run on this larger input.

However, rather than running it as a Jupyter cell (for example, using `%%timeit`), since we're going to compare it with a distributed, Map-Reduce approach in the next part, we want the comparison to be as "fair" as possible. Therefore, you should:

1. Take your `mat_vec_product()` function implementation from above and copy-and-paste it into the indicated part of the following code cell,
2. Run the code cell to save the full contents into a file named `serial_matvec.py`, and
3. Run the code cell **immediately after** to actually execute `serial_matvec.py`.

The code we've provided in the next cell makes sure that `serial_matvec.py` outputs the total runtime (in seconds) for your serial implementation. For reference, the runtime for our solution varies from about `2.4` to `3.9` seconds, with a mean of around `3.75`.

In [ ]:
%%writefile serial_matvec.py
from datetime import datetime, timedelta
import sys
import joblib

### Start Q1.1 Code ###
  
### End Q1.1 Code ###

if __name__ == "__main__":
  start_time = datetime.now()
  A_rand = joblib.load("A_large.pkl")
  b_rand = joblib.load("b_large.pkl")
  result = mat_vec_product(A_rand, b_rand)
  end_time = datetime.now()
  elapsed_time = end_time - start_time
  sys.stderr.write(str(elapsed_time / timedelta(seconds=1)) + "\n")



Overwriting serial_matvec.py


In [ ]:
!python serial_matvec.py

Once `serial_matvec.py` has finished running, it should output an elapsed time in seconds as its last line of output. Copy that runtime into the following cell (as the value of `serial_runtime`, replacing the `None` that's there right now), since we'll use it at the end of the next part to compute the precise speedup we get by moving to a parallelized Map-Reduce approach!

In [ ]:
#| label: Q1.2-serial-runtime
serial_runtime = None

## Part 2: Map\-Reduced Matrix\-Vector Multiplication

In this part, your job is to implement the alternative **Map-Reduce-based** approach to Matrix-Vector multiplication! The following diagram visualizes how you might **decompose** the overall task into (in this case) three individual embarrassingly-parallel subtasks, then combine the results of these subtasks to obtain the end result:

<center>
<img src='https://jjacobs.me/dsan5500-2025/w12/images/matvec2_img.png' width='55%' />
</center>

### Question 2\.1: Sparse Matrix Format

Because of the way Map-Reduce implementations are structured hardware-wise (where the software essentially "chops" an input file into smaller pieces and distributes the pieces to different cores or processes), it turns out that having our matrix in **sparse, row-by-row format** allows the task to get up and running much more quickly than if we use the Python `list` of `list`s format.

So, to achieve this, your task in this question is to complete the `dense_to_sparse()` function started for you in the following code cell. The input `A` is a matrix in the form of a Python `list` of `list`s as described earlier, and the output should be a Python `list` of **`tuple`s**, where each tuple has the form:

```
(row_index, col_index, value)
```

Thus, if the provided matrix was e.g.

$$
\begin{bmatrix}
11 & 0 \\
21 & 22
\end{bmatrix}
$$

the **sparse, row-by-row** data structure returned by `dense_to_sparse()` with that matrix as input would be the following Python `list`:

```
[(0, 0, 11), (1, 0, 21), (1, 1, 22)]
```

In [ ]:
def dense_to_sparse(A):
  n = len(A)
  A_entries = []
  # Your code here: Use two loops (on outer loop for rows and another inner
  # loop for entries within each row) to construct the 3-tuple described above
  # and append it to A_entries.

  return A_entries



Once you've written your implementation, run the following code cell, which compares the output of your implementation to the expected output for the example matrix given above as well as for `A_small` (the sparse-format `A_small_sp` will be saved for use in the next section)

In [ ]:
example_matrix = [[11, 0], [21, 22]]
example_sp = dense_to_sparse(example_matrix)
print(f"example_matrix in sparse format: {example_sp}")
print(f"Expected output: [(0, 0, 11), (1, 0, 21), (1, 1, 22)]")

A_small_sp = dense_to_sparse(A_small)
print(f"A_small in sparse format: {A_small_sp}")
print(f"Expected output: [(0, 0, 3), (0, 1, 2), (1, 1, 4), (1, 2, 1), (2, 0, 2), (2, 2, 1)]")

example_matrix in sparse format: [(0, 0, 11), (1, 0, 21), (1, 1, 22)]
Expected output: [(0, 0, 11), (1, 0, 21), (1, 1, 22)]
A_small in sparse format: [(0, 0, 3), (0, 1, 2), (1, 1, 4), (1, 2, 1), (2, 0, 2), (2, 2, 1)]
Expected output: [(0, 0, 3), (0, 1, 2), (1, 1, 4), (1, 2, 1), (2, 0, 2), (2, 2, 1)]


Once you're confident that your implementation works, run the following code cell, which uses it to sparse-encode both `A_small` and `A_large` and then serializes both to the working directory (since `mrjob`, described below, requires the task's input to be in plaintext `.txt` format)

In [ ]:
#| label: Q2.1-serialize
def serialize_sparse_matrix(mat, fpath):
  with open(fpath, 'w', encoding='utf-8') as outfile:
    for sparse_entry in mat:
      outfile.write(" ".join(str(e) for e in sparse_entry) + "\n")

A_small_sp = dense_to_sparse(A_small)
serialize_sparse_matrix(A_small_sp, "data/A_small.txt")
A_large_sp = dense_to_sparse(A_large)
serialize_sparse_matrix(A_large_sp, "data/A_large.txt")

### Part 2\.2: Map\-Reduce in Serial \(Base Python\)

One nice feature of the map-reduce approach is that Python has built-in (though **serial**) `map()` and `reduce()` functions, meaning, we can develop and test our code on **small input examples** in **serial**, to make sure it works as intended, **before** we then "port" it from base Python to a parallel framework like Ray.

*(Note that, whereas `map()` is available without needing to import anything in both Python 2 and 3, for Python 3 they removed `reduce()` from the set of automatically-available functions, so that we import it from the `functools` library below.)*

Start by running the following cell, which imports `reduce()` and shows an example of how it works (essentially as an "accumulator"), reducing the list of numbers to a single number via repeated addition:

In [ ]:
#| label: Q2.2-reduce-example
from functools import reduce

# The common functional-programming approach, where we use lambda
# rather than giving the reducer an explicit name
print(reduce(lambda x, y: x + y,  [1, 2, 3]))

# The less-common but still functional-programming approach,
# where we explicitly name the reducer function
my_reducer_lambda = lambda x, y: x + y
print(reduce(my_reducer_lambda, [1, 2, 3]))

# The even-less-common but also still functional-programming approach,
# where we define the reducer function the usual way we define
# functions in Python, using def
def my_reducer_function(x, y):
    return x + y
print(reduce(my_reducer_function, [1, 2, 3]))

6
6
6


For the Matrix-Vector Multiplication case, it turns out that this is exactly the "accumulator" **reduce** step we want to use!

If you take a look at the diagram from the beginning of this section, you can see that the final result (the colored circles in the vector on the right) is computed by just **summing up** the second element of each tuple generated by `map()`, after grouping these outputs by the first tuple.

Though we use the term "group" here, and in the following code block (since it's simulating Map-Reduce in a non-parallel fashion), in reality Map-Reduce environments achieve this "grouping" **implicitly** and **automatically**: [Oversimplifying just a little bit], the **first** element of each tuple tells Map-Reduce which **worker** the `map()` outputs should be passed to, so that each worker can just run `reduce()` on their smaller subset of the full set of `map()` outputs.

For Matrix-Vector Multiplication, then, the Map-Reduce environment would take the outputs from `map()` and automatically pass the `(1,x)` tuples to the 1st worker, the `(2,x)` tuples to the 2nd worker, and the `(3,x)` tuples to the 3rd worker.

*(Note how this implies another nice feature of Map-Reduce: it allows monitoring of outputs **as they arrive** – as their `reduce()` steps are completed – so that you can see as quickly as possible whether or not there were errors somewhere in the pipeline. For example, even if your matrix had $100K \times 100K$ rows, you could monitor the output stream, seeing whether the first few results that appear match what you expect, or whether they're all `0`, all `inf`, all `NaN`, etc.)*

Once you've digested this information, please take some time to notice how the code in the following code cell works! It was changed from a question to a provided-answer because of the delayed release, but it will be very helpful to understand how the `map_result` and `reduce_result` variables here are formed, before moving on to using `mrjob` library to code Matrix-Vector Multiplication in a full-on Map-Reduce environment!

In [ ]:
from functools import reduce

def mv_mapper(val_tuple):
  row_index = val_tuple[0]
  col_index = val_tuple[1]
  A_val = val_tuple[2]
  emission_data = (row_index, A_val * b_small[col_index])
  return (row_index, A_val * b_small[col_index])

# The "grouping" step is done manually here since we're in a serial environment.
# It's done *automatically* in distributed Map-Reduce environments like Hadoop
def simulate_mapreduce(A, b):
  n = len(A)
  num_cores = len(b)
  data_in_cores = [[] for i in range(num_cores)]
  for sparse_row in A_small_sp:
    map_result = mv_mapper(sparse_row)
    data_in_cores[map_result[0]].append((map_result[1]))
    print(data_in_cores)
  print([data_in_cores[i] for i in range(num_cores)])
  reduce_result = [reduce(lambda x, y: x + y, data_in_cores[i], 0) for i in range(num_cores)]
  print(reduce_result)

simulate_mapreduce(A_small_sp, b_small)

[[12], [], []]
[[12, 6], [], []]
[[12, 6], [12], []]
[[12, 6], [12, 1], []]
[[12, 6], [12, 1], [8]]
[[12, 6], [12, 1], [8, 1]]
[[12, 6], [12, 1], [8, 1]]
[18, 13, 9]


### Part 2\.3: Your First \(Non\-Simulated\) Map\-Reduce Job\!

The `uv` environment for this assignment ensures that libraries you've already learned, like `numpy` and `joblib`, are pre-installed. The only library we'll be using here that you're *not* already familiar with is a 3rd-party library named [`mrjob`](https://mrjob.readthedocs.io/en/latest/), which is essentially a lightweight easy-to-use... set of "training wheels" that you can use to develop and test Map-Reduce jobs before you send them to the full-on intensive Map-Reduce environments like [Apache Spark](https://spark.apache.org/) or [Hadoop](https://hadoop.apache.org/) that we'll be learning later on.

This section is named "Part 2.3" rather than "Question 2.3" because, there's no code you need to write! Just run the following two code cells to get a sense for how running Map-Reduce jobs via `mrjob` works:

* The first sets up a `FrequencyCounter` class, which has the two requisite pieces of a Map-Reduce task, `mapper()` and `reducer()`:
    * The `mapper()` function takes a **line** of a text file as input and counts three things: the number of **characters** in the line, the number of **words** in the line (the number of space-separated tokens), and the number of **lines** in the line (which only becomes useful once we use `reducer()` to **aggregate** these individual counts).
* The second then actually **launches** the `FrequencyCounter` task... But where does it get a text file from, to compute the character, word, and line counts on?
    * ...It uses the `.py` file itself! If it helps you to gain intuition you can change the **second** command-line argument to be a random text file, like a book from [Project Gutenberg](https://www.gutenberg.org/), but for now we just use `wordfreq_job.py` **twice**: The first time to tell `mrjob` that the `mapper()` and `reducer()` functions are in `wordfreq_job.py`, and the second time to tell `wordfreq_job.py` that it should count the characters, words, and lines in `wordfreq_job.py` itself.

In [ ]:
%%writefile wordfreq_job.py
from mrjob.job import MRJob

class FrequencyCounter(MRJob):
    def mapper(self, _, line):
        yield "chars", len(line)
        yield "words", len(line.split())
        yield "lines", 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == '__main__':
    FrequencyCounter.run()


Overwriting wordfreq_job.py


In [ ]:
!python wordfreq_job.py wordfreq_job.py

### Question 2\.4: Map\-Reduced Matrix\-Vector Multiplication on the Small Example Input

Once you understand that demo, my hope is that this question is not too hard, if you combine the intuition and structure from the above code with the map-reduced matrix-vector multiplication diagram given at the top of this part.



In [ ]:
%%writefile mr_matvec.py
import os
import sys
from datetime import datetime, timedelta
import joblib
from mrjob.job import MRJob

class MatVecProduct(MRJob):
  def mapper(self, _, line):
    if os.path.isfile("b_small.pkl"):
      b = joblib.load("b_small.pkl")
    else:
      b = joblib.load("b_large.pkl")
    line_elts = line.split()
    row_index = int(line_elts[0])
    col_index = int(line_elts[1])
    A_val = float(line_elts[2])
    # Your code here (use yield() in place of return())

  def reducer(self, key, values):
    # Your code here (use yield() in place of return())

if __name__ == '__main__':
  start_time = datetime.now()
  MatVecProduct.run()
  end_time = datetime.now()
  elapsed_time = end_time - start_time
  sys.stderr.write(str(elapsed_time / timedelta(seconds=1)))


Overwriting mr_matvec.py


Now, open the **Terminal** in VSCode, and execute the following command to run the `MatVecProduct` task on the small example input:

```bash
python mr_matvec.py data/A_small.txt --files data/b_small.pkl | sort > data/result_small.txt
```

Once it runs without issue, come back to this notebook and run the following code cell to display the results! The expected output here is:

```
0    18
1    13
2    9
```

*(your spacing may look a bit different, since the Terminal outputs `\t` tab characters while I'm using individual spaces here, but the numbers should be the same)*

In [ ]:
!more result_small.txt

0	18.0
1	13.0
2	9.0


### Part 2\.5: Map\-Reduced Matrix\-Vector Multiplication on the Large Input

And now, let's try it on the randomly-generated matrix and vector, to compare with our serial implementation from Part 1!

Here, since the Map-Reduce code you wrote should work for any valid inputs, there's no new code to write! Just re-run `mr_matvec.py`, this time using the full `data/A_large.txt` file as input rather than `data/A_small.txt` you used before (and, similarly, the included command ensures that `data/b_large.pkl` is copied into each worker's working memory rather than `data/b_small.pkl`).

So, copy the following command and paste it into the Terminal you created for the previous Question, then come back to the notebook like you did before: this time, rather than examining the result itself (which will just be a big list of 2000 numbers, not very interesting to look at on its own), focus on the reported runtime which should be printed out in the Terminal once the Map-Reduce task has finished running, copy-and-paste this number into the code cell below, and read onwards for the takeaway :)

```bash
python mr_matvec.py data/A_large.txt -q --files data/b_large.pkl > data/result_large.txt
```

In [ ]:
map_reduced_runtime = None # Replace with runtime from above `python mr_matvec.py` call
if 'serial_runtime' in globals():
    speedup_ratio = serial_runtime / map_reduced_runtime
    print(f"Speedup of Map-Reduce approach relative to serial approach: {speedup_ratio}")
else:
    print(f"Run earlier cell to compute serial_runtime first!")

If your code is implemented correctly, you should be able to achieve at least a 2x speedup in the Map-Reduced version (sometimes close to 3x). And, note how that speedup is achieved with only a $2500 \times 2500$ matrix: the efficiency increase becomes more and more pronounced for larger matrices, which was precisely one of the key contributors to Google's early success: that they could compute PageRank scores in parallel on large matrices (like the matrix representing all links between webpages on the internet)!

*(See the book mentioned in class, [Lekovec, Rajaraman, and Ullman (2014)](http://infolab.stanford.edu/~ullman/mmds/book.pdf), for more on that... it's legally free online at that link!)*